In [1]:
import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    messages: Annotated[list[str], operator.add]

def reply(state: State) -> dict:
    last = state["messages"][-1]
    return {"messages": [f"Echo: {last}"]}

g = StateGraph(State)
g.add_node("reply", reply)
g.add_edge(START, "reply")
g.add_edge("reply", END)

graph = g.compile(checkpointer=InMemorySaver())

a = {"configurable": {"thread_id" : "a"}}
b = {"configurable": {"thread_id" : "b"}}
graph.invoke({"messages": ["Hello"]}, a)
print("thread a, turn 2:", graph.invoke({"messages": ["second from A"]}, a)["messages"])
print("thread b, turn 1:", graph.invoke({"messages": ["hi from B"]}, b)["messages"])


thread a, turn 2: ['Hello', 'Echo: Hello', 'second from A', 'Echo: second from A']
thread b, turn 1: ['hi from B', 'Echo: hi from B']


In [2]:
snap = graph.get_state(a)
print("get_state values:", snap.values)
print("next nodes      :", snap.next)
print("history (oldest last), step numbers and what was written:")
for s in graph.get_state_history(a):
     print("  step", s.metadata["step"], "| next:", s.next, "| n messages:", len(s.values.get("messages", [])))

get_state values: {'messages': ['Hello', 'Echo: Hello', 'second from A', 'Echo: second from A']}
next nodes      : ()
history (oldest last), step numbers and what was written:
  step 4 | next: () | n messages: 4
  step 3 | next: ('reply',) | n messages: 3
  step 2 | next: ('__start__',) | n messages: 2
  step 1 | next: () | n messages: 2
  step 0 | next: ('reply',) | n messages: 1
  step -1 | next: ('__start__',) | n messages: 0
